In [27]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import torch
from cleanfid import fid
from metrics.cmmd_pytorch.main import compute_cmmd
import torch_fidelity
import os
import numpy as np

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
def calc_cdist_part(features_1, features_2, batch_size=10000):
    dists = []
    for feat2_batch in features_2.split(batch_size):
        dists.append(torch.cdist(features_1, feat2_batch).cpu())
    return torch.cat(dists, dim=1)


def calculate_precision_recall_part(features_1, features_2, neighborhood=3, batch_size=10000):
    # Precision
    dist_nn_1 = []
    for feat_1_batch in features_1.split(batch_size):
        dist_nn_1.append(calc_cdist_part(feat_1_batch, features_1, batch_size).kthvalue(neighborhood + 1).values)
    dist_nn_1 = torch.cat(dist_nn_1)
    precision = []
    for feat_2_batch in features_2.split(batch_size):
        dist_2_1_batch = calc_cdist_part(feat_2_batch, features_1, batch_size)
        precision.append((dist_2_1_batch <= dist_nn_1).any(dim=1).float())
    precision = torch.cat(precision).mean().item()
    # Recall
    dist_nn_2 = []
    for feat_2_batch in features_2.split(batch_size):
        dist_nn_2.append(calc_cdist_part(feat_2_batch, features_2, batch_size).kthvalue(neighborhood + 1).values)
    dist_nn_2 = torch.cat(dist_nn_2)
    recall = []
    for feat_1_batch in features_1.split(batch_size):
        dist_1_2_batch = calc_cdist_part(feat_1_batch, features_2, batch_size)
        recall.append((dist_1_2_batch <= dist_nn_2).any(dim=1).float())
    recall = torch.cat(recall).mean().item()
    return precision, recall

In [19]:

gen_img_path = "data_root/generated/model/c.l4.kv_moodeng-50_lr2.5e-4_f0.5_b1g4/checkpoint-3000/A photo of moodeng/3.00"
ref_img_path = "data_root/data/real_data/moodeng/sd/moodeng-50"
    


metrics_dict = torch_fidelity.calculate_metrics(
    input1=gen_img_path, 
    input2=ref_img_path, 
    cuda=True, 
    isc=True,
    
    feature_extractor="inception-v3-compat",  # critical for PRC

    prc=True, 
    samples_find_deep=False,             # True if images are in subfolders
    verbose=True
)

Creating feature extractor "inception-v3-compat" with features ['logits_unbiased']
Extracting features from input1
Looking for samples non-recursivelty in "data_root/generated/model/c.l4.kv_moodeng-50_lr2.5e-4_f0.5_b1g4/checkpoint-3000/A photo of moodeng/3.00" with extensions png,jpg,jpeg
Found 1000 samples
Processing samples                                                                                                                      
Extracting features from input2
Looking for samples non-recursivelty in "data_root/data/real_data/moodeng/sd/moodeng-50" with extensions png,jpg,jpeg
Found 50 samples, some are lossy-compressed - this may affect metrics
Processing samples                                                                                                                      
Inception Score: 1.4114875917917005 ± 0.08573321828576846


In [33]:


gen_feature_path = "data_root/generated/model/c.l4.kv_crybaby-sd-50_lr2.5e-4_f0.5_b1g4/checkpoint-3000/A photo of a crybaby art toy/1.00/precomputed_features/clip-l-14/03-06-25_07:48_n50.npy"
ref_feature_path = "data_root/data/real_data/crybaby/sd/crybaby-50/precomputed_features/clip-l-14/25-05-25_12:36_n50.npy"


gen_features = torch.tensor(np.load(gen_feature_path))
ref_features = torch.tensor(np.load(ref_feature_path))

precision, recall = calculate_precision_recall_part(ref_features,gen_features,)

print(f"precision:{precision}\nrecall:{recall}")

precision:0.0
recall:0.2800000011920929


In [35]:


gen_feature_path = "data_root/generated/model/c.l4.kv_crybaby-sd-50_lr2.5e-4_f0.5_b1g4/checkpoint-3000/A photo of a crybaby art toy/1.00/precomputed_features/clip-l-14/03-06-25_07:48_n50.npy"
ref_feature_path = "data_root/data/real_data/crybaby/sd/crybaby-50/precomputed_features/clip-l-14/25-05-25_12:36_n50.npy"


gen_features = torch.tensor(np.load(gen_feature_path))
ref_features = torch.tensor(np.load(ref_feature_path))

precision, recall = calculate_precision_recall_part(ref_features,gen_features,neighborhood=5)

print(f"precision:{precision}\nrecall:{recall}")

precision:0.0
recall:0.30000001192092896


In [36]:

gen_feature_path = "data_root/generated/model/c.l4.kv_crybaby-sd-50_lr2.5e-4_f0.5_b1g4/checkpoint-3000/A photo of a crybaby art toy/1.50/precomputed_features/clip-l-14/03-06-25_07:50_n50.npy"
ref_feature_path = "data_root/data/real_data/crybaby/sd/crybaby-50/precomputed_features/clip-l-14/25-05-25_12:36_n50.npy"


gen_features = torch.tensor(np.load(gen_feature_path))
ref_features = torch.tensor(np.load(ref_feature_path))

precision, recall = calculate_precision_recall_part(ref_features,gen_features,)

print(f"precision:{precision}\nrecall:{recall}")


precision, recall = calculate_precision_recall_part(ref_features,gen_features,neighborhood=5)

print(f"precision:{precision}\nrecall:{recall}")

precision:0.07999999821186066
recall:0.07999999821186066
precision:0.11999999731779099
recall:0.1599999964237213


In [37]:
gen_feature_path = "data_root/generated/model/c.l4.kv_crybaby-sd-50_lr2.5e-4_f0.5_b1g4/checkpoint-2500/A photo of a crybaby art toy/3.00/precomputed_features/clip-l-14/03-06-25_07:57_n50.npy"
ref_feature_path = "data_root/data/real_data/crybaby/sd/crybaby-50/precomputed_features/clip-l-14/25-05-25_12:36_n50.npy"


gen_features = torch.tensor(np.load(gen_feature_path))
ref_features = torch.tensor(np.load(ref_feature_path))

precision, recall = calculate_precision_recall_part(ref_features,gen_features,neighborhood=5)

print(f"precision:{precision}\nrecall:{recall}")


precision:0.699999988079071
recall:0.019999999552965164


In [38]:
gen_feature_path = "data_root/generated/model/c.l4.kv_crybaby-sd-50_lr2.5e-4_f0.5_b1g4/checkpoint-3000/A photo of a crybaby art toy/3.00/precomputed_features/clip-l-14/03-06-25_07:57_n50.npy"
ref_feature_path = "data_root/data/real_data/crybaby/sd/crybaby-50/precomputed_features/clip-l-14/25-05-25_12:36_n50.npy"


gen_features = torch.tensor(np.load(gen_feature_path))
ref_features = torch.tensor(np.load(ref_feature_path))

precision, recall = calculate_precision_recall_part(ref_features,gen_features,neighborhood=5)

print(f"precision:{precision}\nrecall:{recall}")


precision:0.6399999856948853
recall:0.11999999731779099


In [39]:
gen_feature_path = "data_root/generated/model/c.l4.kv_crybaby-sd-50_lr2.5e-4_f0.5_b1g4/checkpoint-3000/A photo of a crybaby art toy/4.00/precomputed_features/clip-l-14/03-06-25_08:02_n50.npy"
ref_feature_path = "data_root/data/real_data/crybaby/sd/crybaby-50/precomputed_features/clip-l-14/25-05-25_12:36_n50.npy"


gen_features = torch.tensor(np.load(gen_feature_path))
ref_features = torch.tensor(np.load(ref_feature_path))

precision, recall = calculate_precision_recall_part(ref_features,gen_features,neighborhood=5)

print(f"precision:{precision}\nrecall:{recall}")




precision:0.6399999856948853
recall:0.05999999865889549
